# Week 3 (Part 1): SQl Analysis

**Objective:** use SQL to answer business questions about sales, products, customers and discounts.



In [1]:
import re
import sqlite3
import pandas as pd

pd.options.display.float_format = "{:,.2f}".format
pd.options.display.width = 200

## 1. Designing the database


In [2]:
df = pd.read_csv("../02_Cleaned_Data/superstore_cleaned.csv", parse_dates=["order_date", "ship_date"])
df["order_date"] = df["order_date"].dt.strftime("%Y-%m-%d")   # SQLite stores dates as text
df["ship_date"]  = df["ship_date"].dt.strftime("%Y-%m-%d")

customers = df[["customer_id", "customer_name", "segment"]].drop_duplicates()

products = df[["product_id", "product_name", "category", "sub_category"]].drop_duplicates().reset_index(drop=True)
products.insert(0, "product_key", range(1, len(products) + 1))

orders = df[["order_id", "order_date", "ship_date", "ship_mode", "customer_id",
             "city", "state", "postal_code", "region"]].drop_duplicates()

order_lines = (df.merge(products, on=["product_id", "product_name"])
                 [["row_id", "order_id", "product_key", "sales", "quantity", "discount", "profit"]])

print("customers:", len(customers), "| products:", len(products),
      "| orders:", len(orders), "| order_lines:", len(order_lines))

customers: 793 | products: 1894 | orders: 5009 | order_lines: 9993


## 2. Creating the database and loading the tables

In [3]:
conn = sqlite3.connect("superstore.db")
conn.executescript("""
DROP TABLE IF EXISTS order_lines;
DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS products;
DROP TABLE IF EXISTS customers;

CREATE TABLE customers (
    customer_id   TEXT PRIMARY KEY,
    customer_name TEXT NOT NULL,
    segment       TEXT NOT NULL
);
CREATE TABLE products (
    product_key  INTEGER PRIMARY KEY,
    product_id   TEXT NOT NULL,
    product_name TEXT NOT NULL,
    category     TEXT NOT NULL,
    sub_category TEXT NOT NULL
);
CREATE TABLE orders (
    order_id    TEXT PRIMARY KEY,
    order_date  TEXT NOT NULL,
    ship_date   TEXT NOT NULL,
    ship_mode   TEXT,
    customer_id TEXT NOT NULL REFERENCES customers(customer_id),
    city        TEXT,
    state       TEXT,
    postal_code TEXT,
    region      TEXT
);
CREATE TABLE order_lines (
    row_id      INTEGER PRIMARY KEY,
    order_id    TEXT NOT NULL REFERENCES orders(order_id),
    product_key INTEGER NOT NULL REFERENCES products(product_key),
    sales       REAL NOT NULL,
    quantity    INTEGER NOT NULL,
    discount    REAL NOT NULL,
    profit      REAL NOT NULL
);
""")

for name, table in [("customers", customers), ("products", products),
                    ("orders", orders), ("order_lines", order_lines)]:
    table.to_sql(name, conn, if_exists="append", index=False)
conn.commit()

for name in ["customers", "products", "orders", "order_lines"]:
    print(name, conn.execute(f"SELECT COUNT(*) FROM {name}").fetchone()[0])

customers 793
products 1894
orders 5009
order_lines 9993


In [4]:
# saving the schema as a deliverable
schema = "\n\n".join(row[0] + ";" for row in conn.execute("SELECT sql FROM sqlite_master WHERE type='table'"))
open("schema.sql", "w").write(schema)
print(schema[:400], "...")

CREATE TABLE customers (
    customer_id   TEXT PRIMARY KEY,
    customer_name TEXT NOT NULL,
    segment       TEXT NOT NULL
);

CREATE TABLE products (
    product_key  INTEGER PRIMARY KEY,
    product_id   TEXT NOT NULL,
    product_name TEXT NOT NULL,
    category     TEXT NOT NULL,
    sub_category TEXT NOT NULL
);

CREATE TABLE orders (
    order_id    TEXT PRIMARY KEY,
    order_date  TEXT  ...


## 3. Running the queries

In [6]:
sql_text = open("queries.sql").read()
queries = {int(m.group(1)): m.group(0)
           for m in re.finditer(r"-- Q(\d+):.*?(?=\n-- Q\d+:|\Z)", sql_text, re.S)}
print("Queries found:", sorted(queries))

def q(n, show_sql=True):
    sql = queries[n].strip()
    if show_sql:
        print(sql, "\n")
    return pd.read_sql(sql, conn)

Queries found: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]


## Q1: What are the headline KPIs?
SQL concepts; JOIN, SUM, COUNT DISTINCT, ROUND

In [7]:
r1 = q(1)
r1

-- Q1: What are the headline KPIs?
SELECT ROUND(SUM(ol.sales), 2)                                   AS total_sales,
       ROUND(SUM(ol.profit), 2)                                  AS total_profit,
       COUNT(DISTINCT ol.order_id)                               AS total_orders,
       COUNT(DISTINCT o.customer_id)                             AS total_customers,
       SUM(ol.quantity)                                          AS units_sold,
       ROUND(SUM(ol.profit) * 100.0 / SUM(ol.sales), 2)          AS profit_margin_pct,
       ROUND(SUM(ol.sales) / COUNT(DISTINCT ol.order_id), 2)     AS avg_order_value
FROM order_lines ol
JOIN orders o ON o.order_id = ol.order_id; 



,total_sales,total_profit,total_orders,total_customers,units_sold,profit_margin_pct,avg_order_value
0,"2,296,919.49","286,409.08",5009,793,37871,12.47,458.56


**Finding:** The SQL result matches the Week 2 pandas result exactly: sales $2,296,919.49, profit $286,409.08, 5,009 orders, 793 customers, 37,871 units, a 12.47% margin and an average order value of $458.56. Two different tools giving the same numbers is a good check that both the cleaning and the database are correct.

## Q2: How do sales and profit change by year?
SQL concepts; STRFTIME, GROUP BY, ORDER BY

In [8]:
r2 = q(2)
r2

-- Q2: How do sales and profit change by year?
SELECT STRFTIME('%Y', o.order_date)                              AS year,
       ROUND(SUM(ol.sales), 2)                                   AS sales,
       ROUND(SUM(ol.profit), 2)                                  AS profit,
       COUNT(DISTINCT ol.order_id)                               AS orders
FROM order_lines ol
JOIN orders o ON o.order_id = ol.order_id
GROUP BY year
ORDER BY year; 



,year,sales,profit,orders
0,2014,"483,966.13","49,556.03",969
1,2015,"470,532.51","61,618.60",1038
2,2016,"609,205.60","81,795.17",1315
3,2017,"733,215.26","93,439.27",1687


**Finding:** 2017 was the best year (sales $733K, profit $93K, 1,687 orders). 2015 is the only year in which sales fell, yet profit still rose, from $49.6K to $61.6K.

## Q3: Which five months had the highest sales?
SQL concepts; GROUP BY, ORDER BY DESC, LIMIT

In [9]:
r3 = q(3)
r3

-- Q3: Which five months had the highest sales?
SELECT STRFTIME('%Y-%m', o.order_date)                           AS year_month,
       ROUND(SUM(ol.sales), 2)                                   AS sales
FROM order_lines ol
JOIN orders o ON o.order_id = ol.order_id
GROUP BY year_month
ORDER BY sales DESC
LIMIT 5; 



,year_month,sales
0,2017-11,"118,447.82"
1,2016-12,"96,999.04"
2,2017-09,"87,866.65"
3,2017-12,"83,829.32"
4,2014-09,"81,777.35"


**Finding:** the five best months are November 2017, December 2016, September 2017, December 2017, and September 2014. Every one falls in September, November or December which confirms the seasonal pattern found in Week 2.

## Q4. Which region performs best, and which earns the weakest margin?

SQL concepts; JOIN, GROUP BY, margin calculation


In [13]:
r4 = q(4)
r4

-- Q4: Which region performs best, and which earns the weakest margin?
SELECT o.region,
       ROUND(SUM(ol.sales), 2)                                   AS sales,
       ROUND(SUM(ol.profit), 2)                                  AS profit,
       COUNT(DISTINCT o.order_id)                                AS orders,
       ROUND(SUM(ol.profit) * 100.0 / SUM(ol.sales), 2)          AS margin_pct
FROM order_lines ol
JOIN orders o ON o.order_id = ol.order_id
GROUP BY o.region
ORDER BY sales DESC; 



,region,sales,profit,orders,margin_pct
0,West,"725,457.82","108,418.45",1611,14.94
1,East,"678,499.87","91,534.84",1401,13.49
2,Central,"501,239.89","39,706.36",1175,7.92
3,South,"391,721.91","46,749.43",822,11.93


**Finding:** West leads on sales ($725K) and profit ($108K) with the best margin (14.94%). Central has the weakest margin (7.92%). It sells more than South ($501K against $392K) but earns less profit ($39.7K against $46.7K).

## Q5: How do the three categories compare?

SQL concepts; JOIN, GROUP BY, SUM

In [14]:
r5 = q(5)
r5

-- Q5: How do the three categories compare?
SELECT p.category,
       ROUND(SUM(ol.sales), 2)                                   AS sales,
       ROUND(SUM(ol.profit), 2)                                  AS profit,
       SUM(ol.quantity)                                          AS units,
       ROUND(SUM(ol.profit) * 100.0 / SUM(ol.sales), 2)          AS margin_pct
FROM order_lines ol
JOIN products p ON p.product_key = ol.product_key
GROUP BY p.category
ORDER BY sales DESC; 



,category,sales,profit,units,margin_pct
0,Technology,"836,154.03","145,454.95",6939,17.40
1,Furniture,"741,718.42","18,463.33",8026,2.49
2,Office Supplies,"719,047.03","122,490.80",22906,17.04


**Finding:** Technology has the highest sales ($836K) and profit ($145K). Office Supplies sells by far the most units (22,906). Furniture has almost the same sales as Office Supplies but a margin of only 2.49%.

## Q6: Which sub-categories lose money overall?

SQL concepts; HAVING (filters after grouping)

In [15]:
r6 = q(6)
r6

-- Q6: Which sub-categories lose money overall? (HAVING)
SELECT p.sub_category,
       ROUND(SUM(ol.sales), 2)                                   AS sales,
       ROUND(SUM(ol.profit), 2)                                  AS profit
FROM order_lines ol
JOIN products p ON p.product_key = ol.product_key
GROUP BY p.sub_category
HAVING SUM(ol.profit) < 0
ORDER BY profit; 



,sub_category,sales,profit
0,Tables,"206,965.53","-17,725.48"
1,Bookcases,"114,880.00","-3,472.56"
2,Supplies,"46,673.54","-1,189.10"


**Finding:** Only three sub-categories lose money: Tables (-$17,725 on $206,966 of sales, a margin of about -8.6%), Bookcases (-$3,473) and Supplies (-$1,189). Tables is by far the largest problem, and two of the three are Furniture, which explains Furniture's thin margin.

## Q7: What are the 10 best-selling products?

SQL concepts; JOIN, GROUP BY product name, ORDER BY, LIMIT

In [16]:
r7 = q(7)
r7

-- Q7: What are the 10 best-selling products?
SELECT p.product_name,
       ROUND(SUM(ol.sales), 2)                                   AS sales,
       ROUND(SUM(ol.profit), 2)                                  AS profit
FROM order_lines ol
JOIN products p ON p.product_key = ol.product_key
GROUP BY p.product_name
ORDER BY sales DESC
LIMIT 10; 



,product_name,sales,profit
0,Canon imageCLASS 2200 Advanced Copier,"61,599.82","25,199.93"
1,Fellowes PB500 Electric Punch Plastic Comb Bin...,"27,453.38","7,753.04"
2,Cisco TelePresence System EX90 Videoconferenci...,"22,638.48","-1,811.08"
3,HON 5400 Series Task Chairs for Big and Tall,"21,870.58",0.00
4,GBC DocuBind TL300 Electric Binding System,"19,823.48","2,233.51"
5,GBC Ibimaster 500 Manual ProClick Binding System,"19,024.50",760.98
6,Hewlett Packard LaserJet 3310 Copier,"18,839.69","6,983.88"
7,HP Designjet T520 Inkjet Large Format Printer ...,"18,374.90","4,094.98"
8,GBC DocuBind P400 Electric Binding System,"17,965.07","-1,878.17"
9,High Speed Automatic Electric Letter Opener,"17,030.31",-262.00


**Finding:** The Canon imageCLASS 2200 Advanced Copier sells $61,600, more than double the number two product ($27,453). Of the ten best sellers, three lose money (Cisco TelePresence System EX90 -$1,811, GBC DocuBind P400 -$1,878, High Speed Automatic Electric Letter Opener -$262) and one breaks even. High sales do not guarantee profit.

## Q8: What are the 10 most profitable products?

SQL concepts; JOIN, GROUP BY, ORDER BY, LIMIT

In [17]:
r8 = q(8)
r8

-- Q8: What are the 10 most profitable products?
SELECT p.product_name,
       ROUND(SUM(ol.profit), 2)                                  AS profit,
       ROUND(SUM(ol.sales), 2)                                   AS sales
FROM order_lines ol
JOIN products p ON p.product_key = ol.product_key
GROUP BY p.product_name
ORDER BY profit DESC
LIMIT 10; 



,product_name,profit,sales
0,Canon imageCLASS 2200 Advanced Copier,"25,199.93","61,599.82"
1,Fellowes PB500 Electric Punch Plastic Comb Bin...,"7,753.04","27,453.38"
2,Hewlett Packard LaserJet 3310 Copier,"6,983.88","18,839.69"
3,Canon PC1060 Personal Laser Copier,"4,570.93","11,619.83"
4,HP Designjet T520 Inkjet Large Format Printer ...,"4,094.98","18,374.90"
5,Ativa V4110MDD Micro-Cut Shredder,"3,772.95","7,699.89"
6,"3D Systems Cube Printer, 2nd Generation, Magenta","3,717.97","14,299.89"
7,Plantronics Savi W720 Multi-Device Wireless He...,"3,696.28","9,367.29"
8,Ibico EPK-21 Electric Binding System,"3,345.28","15,875.92"
9,Zebra ZM400 Thermal Label Printer,"3,343.54","6,965.70"


**Finding:** The Canon imageCLASS 2200 Advanced Copier is also the most profitable product ($25,200), more than three times the number two (Fellowes PB500 binding machine, $7,753). The list is dominated by office technology equipment: copiers, printers, binding machines, a shredder and a headset system.

## Q9: Which 10 products lose the most money?
*SQL concepts; JOIN, GROUP BY, AVG, ORDER BY ASC*

In [18]:
r9 = q(9)
r9

-- Q9: Which 10 products lose the most money?
SELECT p.product_name,
       ROUND(SUM(ol.profit), 2)                                  AS profit,
       ROUND(SUM(ol.sales), 2)                                   AS sales,
       ROUND(AVG(ol.discount), 2)                                AS avg_discount
FROM order_lines ol
JOIN products p ON p.product_key = ol.product_key
GROUP BY p.product_name
ORDER BY profit ASC
LIMIT 10; 



,product_name,profit,sales,avg_discount
0,Cubify CubeX 3D Printer Double Head Print,"-8,879.97","11,099.96",0.53
1,Lexmark MX611dhe Monochrome Laser Printer,"-4,589.97","16,829.90",0.40
2,Cubify CubeX 3D Printer Triple Head Print,"-3,839.99","7,999.98",0.50
3,Chromcraft Bull-Nose Wood Oval Conference Tabl...,"-2,876.12","9,917.64",0.28
4,Bush Advantage Collection Racetrack Conference...,"-1,934.40","9,544.73",0.35
5,GBC DocuBind P400 Electric Binding System,"-1,878.17","17,965.07",0.45
6,Cisco TelePresence System EX90 Videoconferenci...,"-1,811.08","22,638.48",0.50
7,Martin Yale Chadless Opener Electric Letter Op...,"-1,299.18","16,656.20",0.10
8,Balt Solid Wood Round Tables,"-1,201.06","6,518.75",0.20
9,BoxOffice By Design Rectangular and Half-Moon ...,"-1,148.44","1,706.25",0.48


**Finding:** The Cubify CubeX 3D Printer (double head) loses $8,880 at an average discount of 53%. Most of the worst products carry average discounts of 40% to 53%. Four of the ten are tables, which matches Q6. The Martin Yale electric letter opener loses $1,299 with only a 10% discount, so discounting is not the only cause; pricing or cost may also be wrong.

## Q10:  Who are the 10 customers with the highest sales?

SQL concepts; Multiple JOINs, GROUP BY, ORDER BY, LIMIT

In [19]:
r10 = q(10)
r10

-- Q10: Who are the 10 customers with the highest sales?
SELECT c.customer_name,
       ROUND(SUM(ol.sales), 2)                                   AS sales,
       ROUND(SUM(ol.profit), 2)                                  AS profit,
       COUNT(DISTINCT o.order_id)                                AS orders
FROM order_lines ol
JOIN orders o    ON o.order_id = ol.order_id
JOIN customers c ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.customer_name
ORDER BY sales DESC
LIMIT 10; 



,customer_name,sales,profit,orders
0,Sean Miller,"25,043.05","-1,980.74",5
1,Tamara Chand,"19,052.22","8,981.32",5
2,Raymond Buch,"15,117.34","6,976.10",6
3,Tom Ashbrook,"14,595.62","4,703.79",4
4,Adrian Barton,"14,473.57","5,444.81",10
5,Ken Lonsdale,"14,175.23",806.85,12
6,Sanjit Chand,"14,142.33","5,757.41",9
7,Hunter Lopez,"12,873.30","5,622.43",6
8,Sanjit Engle,"12,209.44","2,650.68",11
9,Christopher Conant,"12,129.07","2,177.05",5


**Finding:** Sean Miller is the top customer by sales ($25,043) but the business lost $1,981 on his five orders. Tamara Chand ($19,052 in sales) delivered the highest profit of any customer ($8,981). The biggest buyer is not always the best customer.

## Q11:  How does the size of the discount relate to profit?

SQL concepts; CASE WHEN, GROUP BY on a calculated band

In [20]:
r11 = q(11)
r11

-- Q11: How does the size of the discount relate to profit? (CASE WHEN)
SELECT CASE WHEN discount = 0    THEN '1. No discount'
            WHEN discount <= 0.2 THEN '2. Up to 20%'
            WHEN discount <= 0.4 THEN '3. 21% to 40%'
            ELSE                      '4. Over 40%' END         AS discount_band,
       COUNT(*)                                                 AS order_lines,
       ROUND(SUM(sales), 2)                                     AS sales,
       ROUND(SUM(profit), 2)                                    AS profit
FROM order_lines
GROUP BY discount_band
ORDER BY discount_band; 



,discount_band,order_lines,sales,profit
0,1. No discount,4798,"1,087,908.47","320,987.60"
1,2. Up to 20%,3803,"846,522.24","100,785.47"
2,3. 21% to 40%,459,"233,856.53","-35,805.41"
3,4. Over 40%,933,"128,632.25","-99,558.59"


**Finding:** Undiscounted lines earned $320,988 and lines with discounts up to 20% earned $100,785. Lines discounted 21% to 40% lost $35,805 and lines discounted over 40% lost $99,559. Every discount level above 20% loses money overall: 1,392 order lines lost about $135K in total. This is strong evidence for a discount cap.

## Q12: Which customer segment is the most valuable?

SQL concepts; JOIN, GROUP BY, COUNT DISTINCT

In [21]:
r12 = q(12)
r12

-- Q12: Which customer segment is the most valuable?
SELECT c.segment,
       COUNT(DISTINCT c.customer_id)                            AS customers,
       ROUND(SUM(ol.sales), 2)                                  AS sales,
       ROUND(SUM(ol.profit), 2)                                 AS profit,
       ROUND(SUM(ol.sales) / COUNT(DISTINCT c.customer_id), 2)  AS sales_per_customer
FROM order_lines ol
JOIN orders o    ON o.order_id = ol.order_id
JOIN customers c ON c.customer_id = o.customer_id
GROUP BY c.segment
ORDER BY sales DESC; 



,segment,customers,sales,profit,sales_per_customer
0,Consumer,409,"1,161,401.34","134,119.21","2,839.61"
1,Corporate,236,"706,146.37","91,979.13","2,992.15"
2,Home Office,148,"429,371.78","60,310.74","2,901.16"


**Finding:** Consumer is the largest segment (409 customers, $1.16M, about half of all sales). Corporate has the highest sales per customer ($2,992), but the three segments are close ($2,840 to $2,992). The built-in segment label therefore explains little, which is why a behaviour-based segmentation is used in Part B.

## Q13: How many customers are loyal (10 or more orders)?

SQL concepts; CTE (WITH), subquery, filtering on a calculated count

In [22]:
r13 = q(13)
r13

-- Q13: How many customers are loyal (10 or more orders)? (CTE + HAVING)
WITH orders_per_customer AS (
    SELECT customer_id, COUNT(*) AS n_orders
    FROM orders
    GROUP BY customer_id
)
SELECT COUNT(*)                                                 AS customers_with_10_plus_orders,
       (SELECT COUNT(*) FROM orders_per_customer)               AS all_customers,
       ROUND(AVG(n_orders), 1)                                  AS avg_orders_of_these
FROM orders_per_customer
WHERE n_orders >= 10; 



,customers_with_10_plus_orders,all_customers,avg_orders_of_these
0,88,793,11.00


**Finding:** 88 of the 793 customers (11%) placed 10 or more orders, averaging 11 orders each over the four years. This small loyal group is worth protecting.

# 4. Summary of the SQL analysis
- The SQL totals match the pandas totals from Week 2, so the data and the database can be trusted.
- Sales are concentrated in September, November and December, and the business has grown strongly since 2015.
- Furniture has a margin of 2.49%, and Tables, Bookcases and Supplies lose money overall.
- The weakest region is Central (7.92% margin).
- **Discounts above 20% lose money overall** (about $135K across 1,392 lines), and many of the worst products carry discounts of 40% to 53%.
- The biggest customers and biggest products are not always the most profitable.
- 11% of customers (88) placed 10 or more orders.


In [23]:
conn.close()